# 第 3 章 — IRC で反応経路を辿る

**ゴール**
- 2 章で見つけた TS から **IRC (Intrinsic Reaction Coordinate)** を前向き／後ろ向きに流す
- エネルギープロファイルを描いて、本当に HCN ⇄ HNC を繋いでいることを確認する

前提: `examples/02_transition_state_xtb.ipynb` を実行し終わって、`ts.traj` が存在することを想定しています。

In [ ]:
from ase.io import read
from tblite.ase import TBLite
from sella import IRC

# 2 章で収束した TS (trajectory の最終フレーム) を読み込む
ts = read('ts.traj@-1')
ts.calc = TBLite(method='GFN2-xTB', verbosity=0)

print(f'TS エネルギー: {ts.get_potential_energy():.5f} eV')

## 前向き IRC

`direction='forward'` で虚振動の正の方向に降下し、`'reverse'` で負の方向に降下します。
どちらが HCN 側 / HNC 側になるかはたまたまの符号で決まるので、両方流して結果を見ます。

In [ ]:
irc_fwd = IRC(
    ts.copy(),                     # 元の TS を温存するため copy
    trajectory='irc_fwd.traj',
    dx=0.05,                       # IRC イメージ間隔 (Å·√amu)
    eta=1e-4,
    gamma=0.4,
    logfile='irc_fwd.log',
)
irc_fwd.atoms.calc = TBLite(method='GFN2-xTB', verbosity=0)
irc_fwd.run(fmax=0.01, steps=200, direction='forward')

In [ ]:
irc_rev = IRC(
    ts.copy(),
    trajectory='irc_rev.traj',
    dx=0.05,
    eta=1e-4,
    gamma=0.4,
    logfile='irc_rev.log',
)
irc_rev.atoms.calc = TBLite(method='GFN2-xTB', verbosity=0)
irc_rev.run(fmax=0.01, steps=200, direction='reverse')

## エネルギープロファイル

両方向の trajectory を結合して、左から右に「HCN 側 → TS → HNC 側」と並ぶようにプロットします。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase.io import Trajectory

def energies(path):
    return np.array([a.get_potential_energy() for a in Trajectory(path)])

E_fwd = energies('irc_fwd.traj')
E_rev = energies('irc_rev.traj')

# reverse 側を反転して連結 (左端: 反転側の最終点, 中央: TS, 右端: 前向きの最終点)
E_total = np.concatenate([E_rev[::-1], E_fwd[1:]])
s       = np.arange(len(E_total)) - (len(E_rev) - 1)   # 反応座標 (TS=0)

plt.figure(figsize=(6, 4))
plt.plot(s, (E_total - E_total.min()) * 1000, marker='o')
plt.axvline(0, ls='--', c='gray', label='TS')
plt.xlabel('IRC step (TS = 0)'); plt.ylabel('ΔE [meV]')
plt.title('HCN ⇌ HNC の IRC プロファイル (xTB)')
plt.legend(); plt.tight_layout(); plt.show()

## 端点の構造を確認する

両端の構造を見て、それぞれ HCN 側 / HNC 側に降りているかをチェックします。

In [ ]:
endpoint_fwd = read('irc_fwd.traj@-1')
endpoint_rev = read('irc_rev.traj@-1')

for name, a in [('forward 末端', endpoint_fwd), ('reverse 末端', endpoint_rev)]:
    d_CH = a.get_distance(1, 0)   # C-H
    d_NH = a.get_distance(2, 0)   # N-H
    label = 'HCN 側' if d_CH < d_NH else 'HNC 側'
    print(f'{name}: d(C-H)={d_CH:.3f} Å, d(N-H)={d_NH:.3f} Å  → {label}')

## 演習

1. `dx` を `0.02` と `0.10` に変えて、IRC プロファイルの滑らかさがどう変わるか比較してください。
2. IRC の末端構造を 2 章のように再最適化して、エネルギーが完全な HCN / HNC 最小と一致するかを確かめましょう。
3. `keep_going=True` を渡すと内部反復に失敗してもループが続きます。難しい系で試してみてください。

---
次章では **拘束 (Constraints)** を使った緩和スキャンと拘束付き鞍点探索を扱います。